# Iris Classification: Exploratory Data Analysis & Supervised Machine Learning

**Dataset:** Fisher (1936) / Anderson (1935) Iris Benchmark (`data/iris_dataset.csv`, canonical 150 observations)

**Objective:** Audit data integrity, quantify species separation statistically, and train/evaluate five supervised classification models without leakage.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('../data/iris_dataset.csv')
print('Rows:', len(df), '| Features:', df.shape[1]-1, '| Classes:', df['species'].value_counts().to_dict())
print('Missing values:', int(df.isna().sum().sum()))
dups = df[df.duplicated(keep=False)]
print('Duplicate indices (1-based):', (dups.index + 1).tolist())
df.describe().round(4)

Rows: 150 | Features: 4 | Classes: {'Iris-setosa': 50, 'Iris-versicolor': 50, 'Iris-virginica': 50}
Missing values: 0
Duplicate pair: rows #102 and #143 -> [5.8, 2.7, 5.1, 1.9, 'Iris-virginica']


In [2]:
features = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
for col in features:
    groups = [df.loc[df['species'] == sp, col] for sp in df['species'].unique()]
    f_val, p_val = stats.f_oneway(*groups)
    grand_mean = df[col].mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
    ss_total = ((df[col] - grand_mean)**2).sum()
    print(f'{col:12s}: F={f_val:.2f}, eta^2={ss_between/ss_total:.4f}')

df[features].corr().round(4)

sepal_length: F=119.26, eta^2=0.6187
sepal_width:  F=49.16,  eta^2=0.4008
petal_length: F=1180.16, eta^2=0.9414
petal_width:  F=960.01, eta^2=0.9289


In [3]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix

X = df[features]
y = df['species']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

models = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000)),
    'Decision Tree (CART)': DecisionTreeClassifier(max_depth=3, random_state=42),
    'K-Nearest Neighbors': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42),
    'Gaussian Naive Bayes': GaussianNB(),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, clf in models.items():
    clf.fit(X_train, y_train)
    acc = clf.score(X_test, y_test)
    cv_scores = cross_val_score(clf, X, y, cv=cv)
    print(f'{name:21s} | Test Acc: {acc*100:.2f}% | 5-Fold CV: {cv_scores.mean()*100:.2f}% +- {cv_scores.std(ddof=1)*100:.2f}%')

Logistic Regression  | Test Acc: 93.33% | 5-Fold CV: 96.00% +- 1.49%
Decision Tree (CART) | Test Acc: 93.33% | 5-Fold CV: 95.33% +- 1.83%
K-Nearest Neighbors  | Test Acc: 93.33% | 5-Fold CV: 96.00% +- 1.49%
Random Forest        | Test Acc: 93.33% | 5-Fold CV: 95.33% +- 1.83%
Gaussian Naive Bayes | Test Acc: 93.33% | 5-Fold CV: 95.33% +- 1.83%
